In [12]:
import os
import random
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas_datareader.data as data
from sklearn import preprocessing
from collections import deque
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, BatchNormalization
from tensorflow.keras.callbacks import TensorBoard

from IPython.display import display
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


# Get data

In [3]:
def dataset(indices, start_date, end_date):
    # define empty dataframe to merge all subframes
    df = pd.DataFrame()

    for index in indices:
        # collect close prices and volume from yahoo api
        tmp = data.DataReader(name=index, data_source='yahoo', start=start_date, end=end_date)[['Close', 'Volume']]
        # rename columns based on current index
        tmp.rename(columns={'Close':'{}_close'.format(index), 'Volume':'{}_volume'.format(index)}, inplace=True)
        # merge to one dataframe
        df = tmp if len(df) == 0 else df.join(tmp)
    
    return df

INDICES = ['BTC-USD', 'ETH-USD', 'BCH-USD', 'LTC-USD', 'XRP-USD']
START_DATE = dt.datetime.today() - dt.timedelta(days=730)
END_DATE = dt.datetime.today()

df = dataset(INDICES, START_DATE, END_DATE)
display(df)

,BTC-USD_close,BTC-USD_volume,ETH-USD_close,ETH-USD_volume,BCH-USD_close,BCH-USD_volume,LTC-USD_close,LTC-USD_volume,XRP-USD_close,XRP-USD_volume
Date,,,,,,,,,,
2017-12-16,19497.400391,12740599808,696.208984,2165690112,1801.880005,971254976,298.971985,1291379968,0.758642,1334770048
2017-12-17,19140.800781,13314599936,719.974976,2147389952,1862.880005,1020590016,318.718994,1703580032,0.728366,805200000
2017-12-18,19114.199219,14839499776,794.645020,3249230080,2196.639893,2538650112,358.335999,1966599936,0.778407,1342720000
2017-12-19,17776.699219,16894499840,826.822998,4096549888,2805.889893,3913339904,350.252991,2150479872,0.791257,1449670016
2017-12-20,16624.599609,22149699584,819.085999,3969939968,3923.070068,11889600512,314.621002,2104060032,0.775964,1007420032
...,...,...,...,...,...,...,...,...,...,...
2019-12-10,7278.119629,18249031194,146.267044,6859512024,208.299561,1171037351,44.644241,2349745527,0.224302,1191690782
2019-12-11,7217.427246,16350490689,143.608002,7037180048,207.180969,1091099892,43.945698,2507211954,0.222053,1072062151
2019-12-12,7243.134277,18927080224,145.604004,7890383412,207.712036,1276162237,44.082169,2740797931,0.219859,1287697362


# Visualize

In [ ]:
# close prices graph
close_cols = [col for col in df.columns if 'close' in col]
df[close_cols].plot(figsize=(15,6))
plt.show()

In [ ]:
# correlation heatmap
df_corr = df[close_cols].pct_change().corr()

p = plt.figure()

data1 = df_corr.values
fig1 = plt.figure(figsize=(10,8))
ax1 = fig1.add_subplot(111)

heatmap1 = ax1.pcolor(data1, cmap=plt.cm.RdYlGn)
fig1.colorbar(heatmap1)

ax1.set_xticks(np.arange(data1.shape[1]) + 0.5, minor=False)
ax1.set_yticks(np.arange(data1.shape[0]) + 0.5, minor=False)
ax1.invert_yaxis()
ax1.xaxis.tick_top()
column_labels = df_corr.columns
row_labels = df_corr.index
ax1.set_xticklabels(column_labels)
ax1.set_yticklabels(row_labels)
plt.xticks(rotation=90)
heatmap1.set_clim(-1,1)
plt.tight_layout()
plt.show()

# LSTM

In [4]:
def preprocess(df, index_to_predict, future_days_to_predict, days_to_use_for_prediction):
    
    def _classify(current, future):
        return 1 if float(current) < float(future) else 0

    df['future'] = df[index_to_predict+'_close'].shift(-future_days_to_predict)
    df['target'] = df.apply(lambda x: _classify(x[index_to_predict+'_close'], x['future']), axis=1)
    df.dropna(inplace=True)
    df.drop(['future'], axis=1, inplace=True)

    for column in df.columns:
        if column !='target':
            df[column] = df[column].pct_change()
            df[column] = preprocessing.scale(df[column].values)
    df.dropna(inplace=True)

    sequential_data = []
    previous_days = deque(maxlen=days_to_use_for_prediction)
    for i in df.values:
        previous_days.append([n for n in i[:-1]])
        if len(previous_days) == days_to_use_for_prediction:
            sequential_data.append([np.array(previous_days), i[-1]])
            
    buy = []
    sell = []
    
    for seq, target in sequential_data:
        if target == 0:
            sell.append([seq, target])
        else:
            buy.append([seq, target])
            
    minimum = min(len(buy), len(sell))
    
    buy = buy[:minimum]
    sell = sell[:minimum]
    
    sequential_data = buy+sell
    random.shuffle(sequential_data)
    
    X = []
    y = []
    
    for seq, target in sequential_data:
        X.append(seq)
        y.append(target)

    return (np.array(X), np.array(y))

In [18]:
df_train, df_test = np.split(df, [int(0.9*len(df))])

INDEX_TO_PREDICT = 'ETH-USD'
FUTURE_DAYS_TO_PREDICT = 1
DAYS_TO_USE_FOR_PRECIDTION = 5
EPOCHS = 50
BATCH_SIZE = 32

X_train, y_train = preprocess(df_train, INDEX_TO_PREDICT, FUTURE_DAYS_TO_PREDICT, DAYS_TO_USE_FOR_PRECIDTION)
print("Total train data: {} - Class Buy: {}, Class Sell: {}".format(X_train.shape[0], np.count_nonzero(y_train == 1), np.count_nonzero(y_train == 0)))

X_test, y_test = preprocess(df_test, INDEX_TO_PREDICT, FUTURE_DAYS_TO_PREDICT, DAYS_TO_USE_FOR_PRECIDTION)
print("Total train data: {} - Class Buy: {}, Class Sell: {}".format(X_test.shape[0], np.count_nonzero(y_test == 1), np.count_nonzero(y_test == 0)))

model = Sequential([
    LSTM(128, input_shape=(X_train.shape[1:]), activation='tanh', return_sequences=True),
    Dropout(0.2),
    BatchNormalization(),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

optimizer = tf.keras.optimizers.Adam(lr=0.001, decay=1e-6)
model.compile(loss='sparse_categorical_crossentropy', optimizer = optimizer, metrics = ['accuracy'])
logdir = os.path.join("logs", dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard = TensorBoard(log_dir=logdir, histogram_freq=1)
model.fit(X_train, y_train, batch_size=BATCH_SIZE, epochs=EPOCHS, 
          validation_data = (X_test, y_test), callbacks=[tensorboard])


Total train data: 630 - Class Buy: 315, Class Sell: 315
Total train data: 58 - Class Buy: 29, Class Sell: 29
Train on 630 samples, validate on 58 samples
Epoch 1/50
630/630 [==============================] - 4s 6ms/sample - loss: 1.4120 - accuracy: 0.4844 - val_loss: 1.5607 - val_accuracy: 0.4897
Epoch 2/50
630/630 [==============================] - 0s 380us/sample - loss: 1.0502 - accuracy: 0.4892 - val_loss: 1.4535 - val_accuracy: 0.5000
Epoch 3/50
630/630 [==============================] - 0s 368us/sample - loss: 0.9282 - accuracy: 0.4959 - val_loss: 1.3289 - val_accuracy: 0.5000
Epoch 4/50
630/630 [==============================] - 0s 348us/sample - loss: 0.8508 - accuracy: 0.5022 - val_loss: 1.2549 - val_accuracy: 0.5000
Epoch 5/50
630/630 [==============================] - 0s 372us/sample - loss: 0.8077 - accuracy: 0.4984 - val_loss: 1.1632 - val_accuracy: 0.5000
Epoch 6/50
630/630 [==============================] - 0s 360us/sample - loss: 0.7754 - accuracy: 0.5006 - val_loss: 1.

630/630 [==============================] - 0s 429us/sample - loss: 0.6534 - accuracy: 0.5137 - val_loss: 0.6873 - val_accuracy: 0.5069


In [17]:
%tensorboard --logdir logs

Reusing TensorBoard on port 6006 (pid 3071), started 0:59:28 ago. (Use '!kill 3071' to kill it.)